# AbstractComm 1/3: baselines, Phase 1 (warm-up) and Phase 2 (joint bottlenecked SFT)

**Workflow** (one notebook per stage; each ends with a checkpoint and an analysis gate):
1. **This notebook** → `ckpt_sft/`. Read the **gate table** at the end. Don't move on until the `gate` rows PASS.
2. `abscomm_02_distill.ipynb` (Phase 3, self-distillation under channel noise) → `ckpt_distill/`
3. `abscomm_03_grpo.ipynb` (Phase 4, GRPO) → `ckpt_grpo/` plus the final report against the proposal's targets

**Kaggle:** GPU T4, Internet ON. Run with **Save Version → Save & Run All (Commit)** so it survives closing the tab.
Then, in notebook 2, use *Add Input → Notebook Output → this notebook* and it picks up `ckpt_sft` automatically.
Expected runtime is about 2–2.5 h with Qwen2.5-1.5B (about 1 h with 0.5B).

**Fixes relative to the earlier attempts:**
- Gradient reaches the researcher (`y @ E`, not argmax ids), and the codebook E is trainable.
- The question comes first and only the gold paragraphs are used, so the researcher always sees the question.
- The reader gets the question plus the message.
- The loss covers only the answer tokens: the first answer token is included, padding is excluded.
- Evaluation runs on held-out dev data, with shuffled/random-message ablations and same-model baselines.

In [ ]:
!pip install -q -U transformers peft accelerate datasets

In [ ]:
import collections, copy, glob, json, math, os, random, re, string, time
import matplotlib.pyplot as plt
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from peft import LoraConfig, PeftModel, get_peft_model
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

DEVICE, DTYPE = "cuda", torch.float16      # T4: fp16 everywhere (no hardware bf16)
WORK = "/kaggle/working"
N_DEV = 500                                # validation[:N_DEV] is the fixed dev set for every phase
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Protocol + model config. Notebooks 2 and 3 overwrite this from the checkpoint's config.json.
CFG = dict(model="Qwen/Qwen2.5-1.5B-Instruct", K=8, V=26,
           context="gold",                 # "gold" = the 2 supporting paragraphs, "distractor" = all 10
           max_a_len=512, max_ans_len=24, lora_r=16, lora_alpha=32)

def resolve_ckpt(name_or_repo):
    """A local dir, a Kaggle input (Add Input → your previous notebook's output) containing <name>/bottleneck.pt, or a HF repo id."""
    if os.path.isdir(name_or_repo):
        return name_or_repo
    hits = glob.glob(f"/kaggle/input/**/{name_or_repo}/bottleneck.pt", recursive=True)
    return os.path.dirname(hits[0]) if hits else snapshot_download(name_or_repo)

print(torch.cuda.get_device_name(0))

In [ ]:
PHASE = "sft"
CKPT_IN = None                    # fresh start
CKPT_OUT = f"{WORK}/ckpt_{PHASE}"
CFG.update(model="Qwen/Qwen2.5-1.5B-Instruct", K=8, V=26)   # "Qwen/Qwen2.5-0.5B-Instruct" for a fast pilot; 3B needs 4-bit (two fp16 copies don't fit a T4)
N_TRAIN = 8000                    # train[:N_TRAIN]
WARMUP_STEPS = 150                # Phase 1 optimizer steps, reader frozen (0 = skip)
SFT_EPOCHS = 2                    # Phase 2
BS, ACCUM = 4, 4                  # effective batch 16
LR_LORA, LR_BOTTLENECK = 2e-4, 1e-3
TAU_START, TAU_END = 2.0, 0.5     # Gumbel temperature, annealed over Phase 2
LAMBDA_UTIL, LAMBDA_SEP, D_MIN = 0.1, 0.05, 0.3
EVAL_EVERY, N_QUICK = 100, 100    # quick dev check (real vs shuffled message) every N optimizer steps

## Data
- The researcher reads `Question + Context`, with the question **first**, so truncation can only cut context.
- The reader reads only the question plus the message.

In [ ]:
hotpot = load_dataset("hotpotqa/hotpot_qa", "distractor")

def prep(ex):
    gold = set(ex["supporting_facts"]["title"])
    paras = [f"{t}: {' '.join(s)}" for t, s in zip(ex["context"]["title"], ex["context"]["sentences"])
             if CFG["context"] == "distractor" or t in gold]
    return {"question": ex["question"], "answer": ex["answer"], "context": "\n".join(paras)}

def take(split, start, n):
    d = hotpot[split].select(range(start, start + n))
    return d.map(prep, remove_columns=d.column_names).to_dict()   # plain lists (datasets>=4 returns lazy Columns)

def batches(d, bs, shuffle=False):
    idx = list(range(len(d["question"])))
    if shuffle:
        random.shuffle(idx)
    for i in range(0, len(idx), bs):
        yield {k: [v[j] for j in idx[i:i + bs]] for k, v in d.items()}

DEV = take("validation", 0, N_DEV)
QUICK = {k: v[:N_QUICK] for k, v in DEV.items()}
tok = AutoTokenizer.from_pretrained(CFG["model"], padding_side="left")
EOS = tok.convert_tokens_to_ids("<|im_end|>")
print(DEV["question"][0], "→", DEV["answer"][0], "\n", DEV["context"][0][:300], "...")

## The two agents and the discrete channel
```
Researcher A: [Question + Context] [q_1 … q_K] ──LoRA_A──> h at the K query slots ──head──> logits (K × V)
                                                                   Gumbel-softmax, hard (straight-through)
Reader B:     [Question: …\nMessage:] [E[m_1] … E[m_K]] [\nAnswer:] ──LoRA_B──> " answer<|im_end|>"
```
- **K learned query slots**: each message token is read off its own position, not one pooled vector.
- **E**: the shared codebook embedding table (V × d), trainable. It is the only thing that crosses the channel.
- **Straight-through gradient**: one-hot vectors in the forward pass, soft gradient back into A in the backward pass (`y @ E`, never `argmax` ids).
- **Loss only on the answer tokens**: the first answer token and `<|im_end|>` are included, and padding is excluded.

In [ ]:
def positions(mask):
    return (mask.cumsum(-1) - 1).clamp(min=0)

class AbstractComm(nn.Module):
    def __init__(self, ckpt=None):
        super().__init__()
        K, V = CFG["K"], CFG["V"]
        self.A, self.B = self._agent(ckpt, "researcher"), self._agent(ckpt, "reader")
        d = self.A.config.hidden_size
        s = self.A.get_input_embeddings().weight.float().std().item()   # match the scale of real token embeddings
        self.query = nn.Parameter(torch.randn(K, d, device=DEVICE) * s)
        self.head = nn.Linear(d, V, device=DEVICE)
        self.codebook = nn.Parameter(torch.randn(V, d, device=DEVICE) * s)
        if ckpt:
            self.load_state_dict(torch.load(f"{ckpt}/bottleneck.pt", map_location=DEVICE), strict=False)
        self.suffix = tok("\nAnswer:", add_special_tokens=False, return_tensors="pt").input_ids.to(DEVICE)
        self.tau = 1.0

    def _agent(self, ckpt, name):
        base = AutoModelForCausalLM.from_pretrained(CFG["model"], dtype=DTYPE, device_map=DEVICE)
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        base.config.use_cache = False
        if ckpt:
            return PeftModel.from_pretrained(base, f"{ckpt}/{name}", is_trainable=True)
        return get_peft_model(base, LoraConfig(r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=0.05, task_type="CAUSAL_LM",
                                               target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))

    def msg_logits(self, q, ctx, query=None, head=None):
        """Researcher → (B, K, V) codebook logits. query/head overrides are for the frozen GRPO reference policy."""
        query = self.query if query is None else query
        head = self.head if head is None else head
        x = tok([f"Question: {a}\nContext:\n{c}" for a, c in zip(q, ctx)], return_tensors="pt", padding=True,
                truncation=True, max_length=CFG["max_a_len"]).to(DEVICE)
        e = self.A.get_input_embeddings()(x.input_ids)
        e = torch.cat([e, query.to(e.dtype).expand(len(q), -1, -1)], 1)
        m = torch.cat([x.attention_mask, x.attention_mask.new_ones(len(q), CFG["K"])], 1)
        h = self.A.get_base_model().model(inputs_embeds=e, attention_mask=m, position_ids=positions(m), use_cache=False).last_hidden_state
        return head(h[:, -CFG["K"]:].float()).float()

    def embed_msg(self, onehot):
        """(…, K, V) one-hot (or relaxed) message → (…, K, d) reader input embeddings."""
        return onehot.float() @ self.codebook

    def reader_inputs(self, q, msg_emb):
        x = tok([f"Question: {a}\nMessage:" for a in q], return_tensors="pt", padding=True).to(DEVICE)
        E, n = self.B.get_input_embeddings(), len(q)
        e = torch.cat([E(x.input_ids), msg_emb.to(DTYPE), E(self.suffix).expand(n, -1, -1)], 1)
        m = torch.cat([x.attention_mask, x.attention_mask.new_ones(n, CFG["K"] + self.suffix.size(1))], 1)
        return e, m

    def answer_logits(self, e, m, answers):
        """Teacher-forced reader: logits at the answer positions only (memory-light) and their targets."""
        ids = [tok(" " + a + "<|im_end|>", add_special_tokens=False).input_ids[:CFG["max_ans_len"]] for a in answers]
        L = max(map(len, ids))
        a = torch.tensor([i + [tok.pad_token_id] * (L - len(i)) for i in ids], device=DEVICE)
        am = torch.arange(L, device=DEVICE)[None] < torch.tensor([len(i) for i in ids], device=DEVICE)[:, None]
        e = torch.cat([e, self.B.get_input_embeddings()(a)], 1)
        m = torch.cat([m, am.long()], 1)
        h = self.B.get_base_model().model(inputs_embeds=e, attention_mask=m, position_ids=positions(m), use_cache=False).last_hidden_state
        h = h[:, -L - 1:-1][am]                       # position t predicts answer token t+1; the last prompt position predicts the first
        return self.B.get_base_model().lm_head(h).float(), a[am]

    def forward(self, b):
        logits = self.msg_logits(b["question"], b["context"])
        y = F.gumbel_softmax(logits, tau=self.tau, hard=True)
        lg, tgt = self.answer_logits(*self.reader_inputs(b["question"], self.embed_msg(y)), b["answer"])
        return F.cross_entropy(lg, tgt), logits, y

    @torch.no_grad()
    def read(self, q, msg_emb):
        e, m = self.reader_inputs(q, msg_emb)
        with torch.autocast("cuda", dtype=DTYPE):
            out = self.B.generate(inputs_embeds=e, attention_mask=m, max_new_tokens=CFG["max_ans_len"], do_sample=False,
                                  use_cache=True, repetition_penalty=1.0, eos_token_id=[EOS, tok.pad_token_id], pad_token_id=tok.pad_token_id)
        return [s.strip().split("\n")[0] for s in tok.batch_decode(out, skip_special_tokens=True)]

def set_trainable(system, researcher, reader, bottleneck):
    for agent, on in ((system.A, researcher), (system.B, reader)):
        for n, p in agent.named_parameters():
            p.requires_grad = on and "lora_" in n and ".default" in n
    for p in (system.query, system.codebook, *system.head.parameters()):
        p.requires_grad = bottleneck

def make_opt(system, total_steps, lr_lora, lr_bottleneck):
    lora = [p for agent in (system.A, system.B) for n, p in agent.named_parameters() if "lora_" in n and ".default" in n]
    opt = torch.optim.AdamW([{"params": lora, "lr": lr_lora},
                             {"params": [system.query, system.codebook, *system.head.parameters()], "lr": lr_bottleneck}], weight_decay=0.0)
    return opt, get_cosine_schedule_with_warmup(opt, max(1, int(0.05 * total_steps)), total_steps)

scaler = torch.amp.GradScaler("cuda")
def opt_step(opt, sch):
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"] if p.grad is not None], 1.0)
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step()

def util_loss(logits):
    """log V − H(batch-mean codebook usage): 0 iff usage is uniform (penalises collapse)."""
    p = F.softmax(logits, -1).reshape(-1, logits.size(-1)).mean(0)
    return math.log(p.numel()) + (p * p.clamp_min(1e-9).log()).sum()

def sep_loss(y, codebook, d_min):
    """Eq. 3: hinge on the cosine distance between (relaxed) message embeddings of different questions."""
    z = F.normalize((y.float() @ codebook).mean(1), dim=-1)
    off = ~torch.eye(len(z), dtype=torch.bool, device=z.device)
    return F.relu(d_min - (1 - z @ z.T)[off]).mean()

system = AbstractComm(CKPT_IN)
print("trainable LoRA params:", sum(p.numel() for a in (system.A, system.B) for n, p in a.named_parameters() if "lora_" in n))
print(f"GPU memory after load: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Evaluation toolkit (identical in every notebook)

In [ ]:
def norm_ans(s):
    s = "".join(ch for ch in s.lower() if ch not in set(string.punctuation))
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", s).split())

def f1(pred, gold):
    p, g = norm_ans(pred).split(), norm_ans(gold).split()
    same = sum((collections.Counter(p) & collections.Counter(g)).values())
    return 0.0 if same == 0 else 2 * same / (len(p) + len(g))

def score(preds, golds):
    return {"F1": 100 * float(np.mean([f1(p, g) for p, g in zip(preds, golds)])),
            "EM": 100 * float(np.mean([norm_ans(p) == norm_ans(g) for p, g in zip(preds, golds)]))}

def msg_str(m):
    return "".join(f"<{chr(65 + t)}>" if CFG["V"] <= 26 else f"<{t}>" for t in m)

@torch.no_grad()
def get_messages(system, d, bs=16):
    out = []
    for b in batches(d, bs):
        with torch.autocast("cuda", dtype=DTYPE):
            out.append(system.msg_logits(b["question"], b["context"]).argmax(-1))
    return torch.cat(out)

def answer_with(system, d, msgs, bs=32):
    preds = []
    for i, b in enumerate(batches(d, bs)):
        preds += system.read(b["question"], system.embed_msg(F.one_hot(msgs[i * bs:(i + 1) * bs], CFG["V"])))
    return preds

def corrupt(msgs, rate):
    """Channel noise: each token replaced by a uniformly random codebook entry with probability `rate`."""
    return torch.where(torch.rand(msgs.shape, device=msgs.device) < rate, torch.randint_like(msgs, CFG["V"]), msgs)

def codebook_stats(system, msgs):
    V = CFG["V"]
    counts = np.bincount(msgs.flatten().cpu().numpy(), minlength=V)
    p = counts / counts.sum()
    z = F.normalize(system.codebook[msgs].float().mean(1), dim=-1)
    dcos = (1 - z @ z.T)[~torch.eye(len(z), dtype=torch.bool, device=z.device)]
    out = dict(utilization=float((p > 0.01).mean()), usage_entropy_bits=float(-(p[p > 0] * np.log2(p[p > 0])).sum()),
               max_entropy_bits=math.log2(V), distinct_messages=len({tuple(m) for m in msgs.tolist()}), n=len(msgs),
               sep_mean=float(dcos.mean()), sep_p05=float(dcos.float().quantile(0.05)), sep_min=float(dcos.min()), usage=counts.tolist())
    f = np.sort(counts[counts > 0])[::-1].astype(float)
    if len(f) >= 3:   # Fig. 1 test: a power law is linear in log-log, an exponential is linear in log-linear
        r, y = np.arange(1, len(f) + 1), np.log(f)
        def r2_slope(x):
            c = np.polyfit(x, y, 1)
            return float(1 - np.var(y - np.polyval(c, x)) / np.var(y)), float(c[0])
        (out["zipf_r2_powerlaw"], slope), (out["zipf_r2_exponential"], _) = r2_slope(np.log(r)), r2_slope(r)
        out["zipf_alpha"] = -slope
    return out

def evaluate(system, d, conds=("real", "shuffled", "random"), noise=()):
    """real = the researcher's message; shuffled = another example's message; random = uniform codebook tokens."""
    system.eval()
    msgs = get_messages(system, d)
    variants = {"real": msgs, "shuffled": msgs.roll(1, 0), "random": torch.randint_like(msgs, CFG["V"]),
                **{f"noise={r}": corrupt(msgs, r) for r in noise}}
    preds = {c: answer_with(system, d, variants[c]) for c in [*conds, *[f"noise={r}" for r in noise]]}
    res = {c: score(p, d["answer"]) for c, p in preds.items()}
    res["codebook"] = codebook_stats(system, msgs)
    return res, msgs, preds

def quick_eval(system):
    res, _, _ = evaluate(system, QUICK, conds=("real", "shuffled"))
    return {"dev_F1_real": res["real"]["F1"], "dev_F1_shuffled": res["shuffled"]["F1"], "util": res["codebook"]["utilization"]}

SYS_ANS = "Answer the question with a short phrase only (an entity, number, date, or yes/no). No explanation."
SYS_NOTE = "You are a research assistant. Write a brief note (at most 60 words) with the facts a colleague needs to answer the question. Facts only."

@torch.no_grad()
def chat(model, sys_prompt, users, max_new, bs=8):
    out = []
    for i in tqdm(range(0, len(users), bs), desc="baseline", leave=False):
        p = [tok.apply_chat_template([{"role": "system", "content": sys_prompt}, {"role": "user", "content": u}],
                                     tokenize=False, add_generation_prompt=True) for u in users[i:i + bs]]
        x = tok(p, return_tensors="pt", padding=True).to(DEVICE)
        with torch.autocast("cuda", dtype=DTYPE):
            o = model.generate(**x, max_new_tokens=max_new, do_sample=False, use_cache=True, repetition_penalty=1.0,
                               eos_token_id=[EOS, tok.pad_token_id], pad_token_id=tok.pad_token_id)
        out += [s.strip() for s in tok.batch_decode(o[:, x.input_ids.shape[1]:], skip_special_tokens=True)]
    return out

def run_baselines(system, d):
    """Same base model, adapters off. closed-book = floor (the reader without a message); NL two-agent = the H1 reference."""
    system.eval()
    q, c, gold = d["question"], d["context"], d["answer"]
    with system.B.disable_adapter():
        closed = chat(system.B, SYS_ANS, [f"Question: {x}" for x in q], CFG["max_ans_len"])
        full = chat(system.B, SYS_ANS, [f"Question: {x}\nContext:\n{y}" for x, y in zip(q, c)], CFG["max_ans_len"])
        notes = chat(system.B, SYS_NOTE, [f"Question: {x}\nContext:\n{y}" for x, y in zip(q, c)], 96)
        nl = chat(system.B, SYS_ANS, [f"Question: {x}\nNote from researcher: {n}" for x, n in zip(q, notes)], CFG["max_ans_len"])
    first = lambda ps: [p.split("\n")[0] for p in ps]
    return {"closed_book": score(first(closed), gold), "full_context_single_agent": score(first(full), gold),
            "nl_two_agent": score(first(nl), gold), "nl_note_tokens": float(np.mean([len(tok(n).input_ids) for n in notes])),
            "example_notes": notes[:3]}

def gate(res, base, extra=()):
    """'gate' rows must pass before moving to the next phase; 'target' rows are the proposal's final targets (Table 2)."""
    cb, K = res["codebook"], CFG["K"]
    real, nl = res["real"], base["nl_two_agent"]
    rows = [("gate", "message matters: F1(real) − F1(shuffled)", real["F1"] - res["shuffled"]["F1"], ">= 3"),
            ("gate", "beats no-message floor: F1(real) − F1(closed-book)", real["F1"] - base["closed_book"]["F1"], "> 0"),
            ("gate", "codebook utilization (%)", 100 * cb["utilization"], ">= 80"),
            ("target", "H1: F1 retention vs NL two-agent (%)", 100 * real["F1"] / max(nl["F1"], 1e-9), ">= 95"),
            ("target", "H1: EM retention vs NL two-agent (%)", 100 * real["EM"] / max(nl["EM"], 1e-9), ">= 95"),
            ("target", "compression: NL note tokens / K", base["nl_note_tokens"] / K, ">= 4"),
            ("target", "separation: mean cosine distance", cb["sep_mean"], "> 0.3"),
            *extra]
    print(f"{'kind':7s} {'check':55s} {'value':>8s}  {'need':>6s}  result")
    for kind, name, v, need in rows:
        op, t = need.split()
        ok = {">=": v >= float(t), ">": v > float(t), "<=": v <= float(t)}[op]
        print(f"{kind:7s} {name:55s} {v:8.2f}  {need:>6s}  {'PASS' if ok else 'FAIL'}")

def compare(reports):
    print(f"{'phase':10s} {'F1 real':>8s} {'EM real':>8s} {'F1 shuf':>8s} {'F1 rand':>8s} {'util%':>6s} {'sep':>5s} {'msgs':>5s}")
    for ph, r in reports.items():
        d = r["dev"]
        print(f"{ph:10s} {d['real']['F1']:8.2f} {d['real']['EM']:8.2f} {d['shuffled']['F1']:8.2f} {d['random']['F1']:8.2f} "
              f"{100 * d['codebook']['utilization']:6.0f} {d['codebook']['sep_mean']:5.2f} {d['codebook']['distinct_messages']:5d}")

def show_samples(d, msgs, preds, n=10):
    for i in range(n):
        print(f"Q: {d['question'][i]}\n   gold: {d['answer'][i]} | msg: {msg_str(msgs[i].tolist())}\n"
              f"   real → {preds['real'][i]!r} | shuffled → {preds['shuffled'][i]!r}")

def save_ckpt(path, reports, base, log):
    os.makedirs(path, exist_ok=True)
    system.A.save_pretrained(f"{path}/researcher", selected_adapters=["default"])
    system.B.save_pretrained(f"{path}/reader", selected_adapters=["default"])
    torch.save({k: v for k, v in system.state_dict().items() if k.split(".")[0] in ("query", "head", "codebook")}, f"{path}/bottleneck.pt")
    for name, obj in [("config", CFG), ("baselines", base), ("reports", reports), (f"log_{PHASE}", log)]:
        with open(f"{path}/{name}.json", "w") as f:
            json.dump(obj, f, indent=1, default=str)
    print("saved", path)

REPORTS = json.load(open(f"{CKPT_IN}/reports.json")) if CKPT_IN else {}
BASE = json.load(open(f"{CKPT_IN}/baselines.json")) if CKPT_IN else None

## 0. Baselines (same base model, adapters off)
- **closed-book**: the floor. It's what the reader can do with the question alone, so the message has to beat this.
- **NL two-agent**: the H1 reference (the researcher writes a note, the reader answers from question + note). Its note length sets the compression ratio.
- **full-context single agent**: the ceiling.

In [ ]:
BASE = run_baselines(system, DEV)
print(json.dumps(BASE, indent=1))

In [ ]:
TRAIN = take("train", 0, N_TRAIN)
LOG = []

def sft(n_steps, tau_at, lam_util, lam_sep, tag):
    opt, sch = make_opt(system, n_steps, LR_LORA, LR_BOTTLENECK)
    step = micro = 0
    pbar = tqdm(total=n_steps, desc=tag)
    system.train()
    while step < n_steps:
        for b in batches(TRAIN, BS, shuffle=True):
            system.tau = tau_at(step / n_steps)
            with torch.autocast("cuda", dtype=DTYPE):
                ce, logits, y = system(b)
                lu, ls = util_loss(logits), sep_loss(y, system.codebook, D_MIN)
                loss = ce + lam_util * lu + lam_sep * ls
            scaler.scale(loss / ACCUM).backward()
            micro += 1
            if micro % ACCUM:
                continue
            opt_step(opt, sch)
            step += 1
            pbar.update()
            LOG.append(dict(phase=tag, step=step, ce=ce.item(), util=lu.item(), sep=ls.item(), tau=system.tau))
            if step % EVAL_EVERY == 0 or step == n_steps:
                LOG[-1].update(quick_eval(system))
                system.train()
                r = LOG[-1]
                tqdm.write(f"[{tag} {step}/{n_steps}] ce={r['ce']:.3f} util={r['util']:.3f} tau={r['tau']:.2f} | "
                           f"dev F1 real={r['dev_F1_real']:.1f} shuffled={r['dev_F1_shuffled']:.1f} | codebook used={100 * r['util']:.0f}%")
            if step == n_steps:
                break
    pbar.close()

## Phase 1: researcher warm-up (reader frozen)
The reader's LoRA is frozen. Only the researcher's LoRA, the query slots, the head and the codebook E learn.
This forces the information into the message instead of letting the reader memorise answers.

**Checkpoint 1:** dev F1(real) should start pulling away from F1(shuffled). If CE is flat after 150 steps, raise `LR_BOTTLENECK`.

In [ ]:
if WARMUP_STEPS:
    set_trainable(system, researcher=True, reader=False, bottleneck=True)
    sft(WARMUP_STEPS, lambda f: 1.0, LAMBDA_UTIL, 0.0, "phase1_warmup")

## Phase 2: joint bottlenecked SFT
$\mathcal{L} = \mathcal{L}_{answer} + \lambda_1\mathcal{L}_{util} + \lambda_2\mathcal{L}_{sep}$, with τ annealed from 2.0 to 0.5. Everything trains.

In [ ]:
set_trainable(system, researcher=True, reader=True, bottleneck=True)
sft(SFT_EPOCHS * N_TRAIN // (BS * ACCUM), lambda f: TAU_START * (TAU_END / TAU_START) ** f, LAMBDA_UTIL, LAMBDA_SEP, "phase2_sft")
CFG["n_train_used"] = N_TRAIN
save_ckpt(CKPT_OUT, REPORTS, BASE, LOG)       # save before the long eval, in case it crashes

## Analysis: is SFT good enough to move on?

In [ ]:
dev_res, dev_msgs, dev_preds = evaluate(system, DEV)
train_res, _, _ = evaluate(system, {k: v[:300] for k, v in TRAIN.items()}, conds=("real",))
REPORTS[PHASE] = {"dev": dev_res, "train_seen": train_res}

print("baselines (dev):", {k: v for k, v in BASE.items() if k != "example_notes"})
print("AbstractComm (dev):", {c: dev_res[c] for c in ("real", "shuffled", "random")})
print("codebook:", {k: v for k, v in dev_res["codebook"].items() if k != "usage"}, "\n")
gate(dev_res, BASE, extra=[("gate", "memorisation gap: F1(train) − F1(dev)", train_res["real"]["F1"] - dev_res["real"]["F1"], "<= 15")])

In [ ]:
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
def rolling(x, w=25):
    x = np.asarray(x, float)
    return np.convolve(x, np.ones(w) / w, "valid") if len(x) > w else x

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
idx = {tag: [i for i, r in enumerate(LOG) if r["phase"] == tag] for tag in ("phase1_warmup", "phase2_sft")}
for (tag, ii), col in zip(idx.items(), (BLUE, ORANGE)):
    if ii:
        y = rolling([LOG[i]["ce"] for i in ii])
        axes[0].plot(ii[len(ii) - len(y):], y, color=col, lw=2, label=tag)
axes[0].set(title="Answer cross-entropy (rolling mean)", xlabel="optimizer step", ylabel="CE (nats)")
axes[0].legend(frameon=False)

ev = [i for i, r in enumerate(LOG) if "dev_F1_real" in r]
axes[1].plot(ev, [LOG[i]["dev_F1_real"] for i in ev], color=BLUE, lw=2, marker="o", ms=5, label="real message")
axes[1].plot(ev, [LOG[i]["dev_F1_shuffled"] for i in ev], color=ORANGE, lw=2, marker="o", ms=5, label="shuffled message")
axes[1].axhline(BASE["closed_book"]["F1"], color=GRAY, lw=1.5, ls="--", label="closed-book baseline")
axes[1].set(title=f"Dev F1 during training (first {N_QUICK} dev questions)", xlabel="optimizer step", ylabel="F1")
axes[1].legend(frameon=False)

counts = np.sort(np.array(dev_res["codebook"]["usage"]))[::-1]
counts = counts[counts > 0]
axes[2].loglog(np.arange(1, len(counts) + 1), counts, color=BLUE, lw=2, marker="o", ms=5)
z = dev_res["codebook"]
axes[2].set(title=f"Codebook rank-frequency (power-law R²={z.get('zipf_r2_powerlaw', float('nan')):.2f}, "
                  f"exp R²={z.get('zipf_r2_exponential', float('nan')):.2f})", xlabel="token rank", ylabel="uses on dev")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{CKPT_OUT}/training.png", dpi=150)
plt.show()

In [ ]:
show_samples(DEV, dev_msgs, dev_preds)
save_ckpt(CKPT_OUT, REPORTS, BASE, LOG)

## If a gate fails, tweak before moving on
| symptom | likely cause | tweak |
|---|---|---|
| real ≈ shuffled, utilization low | codebook collapse | `LAMBDA_UTIL` → 0.3, `TAU_END` → 1.0, raise `LR_BOTTLENECK` |
| real ≈ shuffled, utilization fine | reader answers from the question and ignores the channel | `WARMUP_STEPS` → 400, `K` → 16, `LR_LORA` → 1e-4 |
| real below closed-book | reader LoRA damaged or overfit | lower `LR_LORA`, `SFT_EPOCHS` → 1 |
| memorisation gap > 15 | too little data | raise `N_TRAIN` (16k+), fewer epochs |
| CE flat from the start | LR too low, or the channel isn't learning | check that the dev F1 curve moves; raise `LR_BOTTLENECK` |

H1 retention and compression are **final** targets. They're expected to improve in Phases 3–4, so don't block on them here.

## Optional: push the checkpoint to the HF Hub
Needs an `HF_TOKEN` Kaggle secret with write access. The repo is created **private**.
The other way to hand off is Kaggle's *Add Input → Notebook Output* in the next notebook; `resolve_ckpt` finds it automatically.

In [ ]:
HF_REPO = None   # e.g. "your-username/abscomm-" + PHASE
if HF_REPO:
    from huggingface_hub import HfApi, login
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    HfApi().create_repo(HF_REPO, private=True, exist_ok=True)
    HfApi().upload_folder(folder_path=CKPT_OUT, repo_id=HF_REPO)
    print("pushed to", HF_REPO)